In [0]:
# Databricks notebook source
# -----------------------------------------------------------------------------
# Small control table: one row per (source, symbol). Reading/writing this is
# cheap regardless of how large the underlying price/NAV tables get.
# -----------------------------------------------------------------------------
from pyspark.sql import functions as F
from delta.tables import DeltaTable


def get_watermark_dict(spark, table_full_name: str, source: str) -> dict:
    """{symbol: last_loaded_date} for one source."""
    if not spark.catalog.tableExists(table_full_name):
        return {}
    df = spark.sql(f"""
        SELECT symbol, last_loaded_date
        FROM {table_full_name}
        WHERE source = '{source}'
    """)
    return {row.symbol: row.last_loaded_date for row in df.collect()}


def update_watermark(spark, table_full_name: str, source: str, symbol_max_dates: dict):
    """symbol_max_dates: {symbol: date} computed in-memory from the batch just
    written — never recomputed by scanning the target table."""
    if not symbol_max_dates:
        return

    rows = [(source, sym, d) for sym, d in symbol_max_dates.items()]
    updates_df = (
        spark.createDataFrame(rows, ["source", "symbol", "last_loaded_date"])
        .withColumn("updated_at", F.current_timestamp())
    )

    target = DeltaTable.forName(spark, table_full_name)
    (
        target.alias("t")
        .merge(updates_df.alias("s"), "t.source = s.source AND t.symbol = s.symbol")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )